In [1]:
import numpy as np
import pandas as pd

# one row per player-season (output of 01_download)
df = pd.read_csv("data/processed/player_seasons_raw.csv")

# total attempts = volume behind each shooting % (approximate: per-game values are rounded)
df["FG3A_total"] = df["FG3A"] * df["GP"]
df["FTA_total"]  = df["FTA"] * df["GP"]

# share of rows with fewer than t total attempts
for t in [1, 10, 20, 30, 50]:
    print(t,
          round((df["FG3A_total"] < t).mean(), 3),
          round((df["FTA_total"] < t).mean(), 3))

1 0.11 0.051
10 0.232 0.182
20 0.299 0.263
30 0.342 0.329
50 0.412 0.439


In [2]:
df["season_year"] = df["SEASON"].str[:4].astype(int)   # "2022-23" -> 2022

In [3]:
# lookup: (player, season) -> stats
# add_shifted: value from k seasons back (k > 0) or ahead (k < 0)
lookup = df[["PLAYER_ID", "season_year", "PTS", "GP", "MIN"]]

def add_shifted(df, k, column, name):
    s = lookup[["PLAYER_ID", "season_year", column]]
    # relabel years by +k: the row at season N receives the value from season N-k
    s = s.assign(season_year=s["season_year"] + k)
    s = s.rename(columns={column: name})
    return df.merge(s, on=["PLAYER_ID", "season_year"],
                    how="left", validate="one_to_one")

In [4]:
# ppg_next: next-season PPG (only used to build the target, never a feature)
df = add_shifted(df, -1, "PTS", "ppg_next")
# lag features: PPG from 1, 2 and 3 seasons back
for k in [1, 2, 3]:
    df = add_shifted(df, k, "PTS", f"ppg_lag{k}")

# next-season games and minutes (only used to filter noisy targets)
df = add_shifted(df, -1, "GP",  "gp_next")
df = add_shifted(df, -1, "MIN", "min_next")

# target: change in PPG from season N to N+1
df["target"] = df["ppg_next"] - df["PTS"]

In [5]:
# NaN share: lags are missing for rookies; target is NaN for 2025-26 (no next season yet)
print(df[["PTS", "ppg_lag1", "ppg_lag2", "ppg_lag3", "ppg_next", "target"]].isna().mean())
print(df["target"].isna().groupby(df["SEASON"]).mean())
print(df["target"].describe())

PTS         0.000000
ppg_lag1    0.271103
ppg_lag2    0.439499
ppg_lag3    0.569576
ppg_next    0.271103
target      0.271103
dtype: float64
SEASON
2013-14    0.180498
2014-15    0.195122
2015-16    0.189076
2016-17    0.193416
2017-18    0.237037
2018-19    0.245283
2019-20    0.177694
2020-21    0.166667
2021-22    0.284298
2022-23    0.165121
2023-24    0.218531
2024-25    0.184534
2025-26    1.000000
Name: target, dtype: float64
count    5060.000000
mean       -0.051838
std         3.300841
min       -19.900000
25%        -2.100000
50%        -0.100000
75%         1.900000
max        15.500000
Name: target, dtype: float64


In [6]:
# sanity check: lags must follow calendar years (see the gap for Klay Thompson)
cols = ["SEASON", "PTS", "ppg_lag1", "ppg_lag2", "ppg_lag3", "ppg_next", "target"]
print(df[df["PLAYER_NAME"] == "LeBron James"][cols])
print(df[df["PLAYER_NAME"] == "Klay Thompson"][cols])

       SEASON   PTS  ppg_lag1  ppg_lag2  ppg_lag3  ppg_next  target
287   2013-14  27.1       NaN       NaN       NaN      25.3    -1.8
783   2014-15  25.3      27.1       NaN       NaN      25.3     0.0
1265  2015-16  25.3      25.3      27.1       NaN      26.4     1.1
1742  2016-17  26.4      25.3      25.3      27.1      27.5     1.1
2261  2017-18  27.5      26.4      25.3      25.3      27.4    -0.1
2806  2018-19  27.4      27.5      26.4      25.3      25.3    -2.1
3333  2019-20  25.3      27.4      27.5      26.4      25.0    -0.3
3868  2020-21  25.0      25.3      27.4      27.5      30.3     5.3
4449  2021-22  30.3      25.0      25.3      27.4      28.9    -1.4
5012  2022-23  28.9      30.3      25.0      25.3      25.7    -3.2
5583  2023-24  25.7      28.9      30.3      25.0      24.4    -1.3
6149  2024-25  24.4      25.7      28.9      30.3      20.9    -3.5
6724  2025-26  20.9      24.4      25.7      28.9       NaN     NaN
       SEASON   PTS  ppg_lag1  ppg_lag2  ppg_lag

In [7]:
# largest increase: tiny sample (1 game next season)
df.loc[df["target"].idxmax()]

PLAYER_ID         1629663
SEASON            2021-22
PLAYER_NAME    Louis King
TEAM_ID        1610612758
TEAM_COUNT              1
AGE                  23.0
GP                     10
MIN                  10.4
PTS                   4.5
FGA                   4.7
FG_PCT              0.319
FG3A                  2.7
FG3_PCT             0.296
FTA                   1.0
FT_PCT                0.7
USG_PCT             0.195
TS_PCT              0.438
PACE                104.7
FG3A_total           27.0
FTA_total            10.0
season_year          2021
ppg_next             20.0
ppg_lag1              7.3
ppg_lag2              2.0
ppg_lag3              NaN
gp_next               1.0
min_next             28.8
target               15.5
Name: 4456, dtype: object

In [8]:
# largest drop: injury (1 game next season)
df.loc[df["target"].idxmin()]

PLAYER_ID              202330
SEASON                2016-17
PLAYER_NAME    Gordon Hayward
TEAM_ID            1610612762
TEAM_COUNT                  1
AGE                      27.0
GP                         73
MIN                      34.5
PTS                      21.9
FGA                      15.8
FG_PCT                  0.471
FG3A                      5.1
FG3_PCT                 0.398
FTA                       5.9
FT_PCT                  0.844
USG_PCT                 0.269
TS_PCT                  0.595
PACE                    92.39
FG3A_total              372.3
FTA_total               430.7
season_year              2016
ppg_next                  2.0
ppg_lag1                 19.7
ppg_lag2                 19.3
ppg_lag3                 16.2
gp_next                   1.0
min_next                  5.3
target                  -19.9
Name: 1613, dtype: object

In [9]:
# total minutes played in N+1
df["min_next_total"] = df["min_next"] * df["gp_next"]

valid = df.dropna(subset=["min_next_total"])   # only players with a next season

for t in [0, 25, 50, 100, 200, 500]:
    d = valid[valid["min_next_total"] >= t]
    print(t, d["target"].min(), d["target"].max())         # range of the target left after each cutoff

0 -19.9 15.5
25 -15.4 15.5
50 -15.4 15.3
100 -15.4 15.3
200 -15.4 14.3
500 -13.7 14.3


In [10]:
# total minutes played in N
df["min_total"] = df["MIN"] * df["GP"]
# rows with a usable next season (>= 200 min in N+1)
with_next = df[df["min_next_total"] >= 200]

# baseline error (no change) by minutes played in N: low minutes are noisier
for lo, hi in [(0, 100), (100, 300), (300, 600), (600, 1000), (1000, 2000), (2000, 4000)]:
    d = with_next[(with_next["min_total"] >= lo) & (with_next["min_total"] < hi)]
    print(lo, hi, len(d), round(d["target"].abs().mean(), 2))

0 100 133 3.71
100 300 254 2.74
300 600 407 2.38
600 1000 624 2.42
1000 2000 1877 2.45
2000 4000 1131 2.47


In [11]:
# save the full feature table (no filters); filtering and splits happen in 03
df.to_csv("data/processed/player_seasons_features.csv", index=False)